In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os
from pathlib import Path

project_root = Path(os.getcwd()).parent if 'spaghetti_code' in os.getcwd() else Path(os.getcwd())
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from cs_priors.metrics.count_sparsity import (
    noise_threshold, source_energies, active_sources, detection_scores, relative_error,
) 
from cs_priors.plotting.plot_complex import plot_matrices
from cs_priors.plotting.plotting import plot_equation

In [ ]:
# Dummy data: 5 sources, 8 freq bins
# Sources 1 and 3 are active in X_true
S, N = 5, 8
X_true = np.zeros((S, N), dtype=complex)
X_true[1, :] = np.random.randn(N) + 1j * np.random.randn(N)
X_true[3, :] = 0.5 * (np.random.randn(N) + 1j * np.random.randn(N))

# X_pred: correctly finds source 1, misses source 3, falsely detects source 0
X_pred = np.zeros((S, N), dtype=complex)
X_pred[0, :] = 0.3 * (np.random.randn(N) + 1j * np.random.randn(N))  # false positive
X_pred[1, :] = X_true[1, :] + 0.05 * np.random.randn(N)              # true positive
# source 3 missing -> false negative

plot_matrices([X_true, X_pred], titles=["X_true", "X_pred"], show_values=False)

In [ ]:
tol = noise_threshold(X_true)
scores = detection_scores(X_true, X_pred, tol=tol)

print(f"Threshold: {tol:.4f}")
print(f"True active:      {active_sources(X_true, tol).tolist()}")
print(f"Predicted active: {active_sources(X_pred, tol).tolist()}")
print()
for k, v in scores.items():
    print(f"  {k:>10s}: {v:.3f}")
print(f"  {'rel_error':>10s}: {relative_error(X_true, X_pred):.3f}")

In [ ]:
# Wrong-prediction vector: -1 for FP or FN sources, 0 otherwise
true_set = set(active_sources(X_true, tol).tolist())
pred_set = set(active_sources(X_pred, tol).tolist())
wrong_set = (true_set - pred_set) | (pred_set - true_set) # union of fn and fp

wrong_matrix = np.zeros((S, 1))
for i in wrong_set:
    wrong_matrix[i, 0] = -1.0

plot_equation(
    X_true, X_pred, wrong_matrix,
    titles=["X_true", "X_pred", f"{len(wrong_set)} wrong"],
    show_values=False,
)

## 2. Metrics on a single vector
Demonstrate `noise_threshold`, `source_energies`, `active_sources`, `detection_scores`, `relative_error`.

In [ ]:
# noise_threshold: factor * max per-source energy ‖X[i,:]‖₂
tol = noise_threshold(X_true)
print(f"Threshold (10% of max source energy): {tol:.4f}")
plot_matrices([X_true, 1j*source_energies(X_true)], dpi=70)
plot_matrices([X_pred, 1j*source_energies(X_pred)], dpi=70)